# 03b · Guardrails over the saved judge output: rule applicability and self-consistency

The judge in `03_compliance_check.ipynb` is a language model, and it has two habits a rule engine would not have: it applies a rule to a drawing that has nothing for the rule to check (the lockout/tagout rule on a pipe isometric that only says "FROM K-1082 NOZZLE N3"), and it sometimes lists a "violation" whose own explanation says the requirement is met. This notebook removes both **deterministically, without a single new AI call**. It reads the judge's answers exactly as saved in `bluerridge_gold_pm.compliance_judge_raw` (1,200 answers from job run 726850339672013, the run recorded in `03_compliance_check.ipynb`, whose AI step is the only one that bills), applies two guardrails, and rewrites `compliance_findings`.

1. **Rule applicability.** Which of the five citations apply to which drawing type, a named constant `compliance_lib.APPLICABILITY` with a one-line engineering rationale per cell (the table below). A real rule engine routes rules to document types before it calls a model; this is that routing, applied after the fact to the saved answers. A FAIL entry whose citation does not apply to the drawing's type is suppressed (`suppressed_by = rule_applicability`).
2. **Self-contradiction.** A FAIL entry is suppressed (`suppressed_by = self_contradiction`) when the judge's own output admits compliance: its structured `found_in_text` is true, or its structured `expected_item`, when it is a tag (such as LOTO-2505A or PSV-2102, not a quoted requirement like \"BW with 100% RT\"), is in the drawing text although it answered `found_in_text: false` (the O/0 confusion of the parser is allowed for), or the lockout/tagout rule was applied to a vessel (which the rule's own text excludes), or its reason contains one of seven conservative phrases such as "is present in the drawing" or "not a violation" (the "is present", "is compliant" and "so it is protected" phrases only count in a sentence that is not negated, so "No LOTO-2702 tag is present" stays a violation). The structured fields are checked first; the phrases are the fallback. Unit-tested in `tests/test_guardrails.py`.

**Nothing is silently dropped.** Every suppressed entry is written to `bluerridge_gold_pm.compliance_suppressed` with the finding's columns plus `suppressed_by` and `suppression_detail`. A drawing whose FAIL entries are all suppressed gets the PASS sentinel row, by the usual grain rules (N FAIL rows, or one PASS, or one REVIEW).

**What is not fixed.** The parser sometimes reads LOTO as ROTO-, LOT0- or LOTOTO-. A judge that flags such a tag is reporting what the text says; no guardrail here guesses the tag back, so those false alarms stay.

**Why a separate notebook.** `03_compliance_check.ipynb` keeps the evidence of the AI run exactly as it ran: its outputs (the raw judge's numbers, row counts, the run id) match its code. This notebook is the fourth task of the doc-intelligence job, after `compliance_check`, and rebuilds `compliance_findings` from the saved answers; it makes no AI call and is safe to re-run.

In [1]:
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql import Window

sys.path.insert(0, os.getcwd())  # a notebook's working directory is its own folder, where compliance_lib.py lives
import compliance_lib as lib

dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
catalog = dbutils.widgets.get("catalog")
prefix = dbutils.widgets.get("schema_prefix")
MAX_ROWS, SUFFIX = 0, ""  # the cells reused from 03_compliance_check read these; this notebook always covers every drawing

SILVER = f"{catalog}.{prefix}_silver_engineering"
GOLD = f"{catalog}.{prefix}_gold_pm"
PARSED = f"{SILVER}.drawings_parsed"
EXTRACTED = f"{SILVER}.drawings_extracted"
GROUND_TRUTH_TABLE = f"{SILVER}.ground_truth_findings"
RAW = f"{GOLD}.compliance_judge_raw"
TARGET = f"{GOLD}.compliance_findings"
SUPPRESSED = f"{GOLD}.compliance_suppressed"
MANIFEST = Path("../01_synthetic_data/outputs/drawing_manifest.json")  # same relative path in the repo and in a bundle deploy
OUT = Path("outputs")  # next to this notebook


def sql_literal(text):
    # `text` as a SQL string literal (backslashes and single quotes escaped).
    return "'" + text.replace("\\", "\\\\").replace("'", "\\'") + "'"


print(f"judge answers (read, never written): {RAW}")
print(f"target: {TARGET}; suppressed entries: {SUPPRESSED}")

judge answers (read, never written): mfg_mc_se_sa.bluerridge_gold_pm.compliance_judge_raw
target: mfg_mc_se_sa.bluerridge_gold_pm.compliance_findings; suppressed entries: mfg_mc_se_sa.bluerridge_gold_pm.compliance_suppressed


## Which rules apply to which drawing type

The constant `compliance_lib.APPLICABILITY`, shown here as the notebook generated it. The cells follow what each drawing type depicts in EPC practice (isometrics carry the line and its weld map; P&IDs carry equipment, relief devices and line classes but no weld joints and no buildings; general arrangements carry equipment locations and exits; specification sheets are tables of classes, welds and relief devices; legend and detail sheets describe no installed item). They were written from engineering practice before looking at how the judge did, not adjusted cell by cell to the ground truth.

| drawing type | flange rating §4.2 | LOTO 1910.147 | egress IBC 1005.1 | weld §341 | relief §322 |
|---|---|---|---|---|---|
| **ISO** | yes: An isometric shows one line with its flanged joints and the line's design pressure, so flange ratings can be checked. | no: An isometric shows pipe, fittings and nozzle tie-ins only; the powered equipment and its isolation tags are on the P&ID and the plan. | no: Means of egress belongs to building plans; an isometric of a pipe run has no exits. | yes: An isometric carries the weld map: each shop and field weld with its joint type and examination. | no: Vessels and their relief devices are shown on the P&ID and the relief schedule, not on a single-line isometric. |
| **PID** | yes: A P&ID tags each line with a flange class and design pressure. | yes: Pumps, compressors and their isolation (LOTO) points are drawn on the P&ID. | no: A process schematic has no building, exits or occupant load. | no: A P&ID is not to scale and does not show weld joints or examination; those are on isometrics and the piping class spec. | yes: Vessels and the PSVs that protect them, with set pressures, are shown on the P&ID. |
| **GA** | no: A general arrangement plan locates equipment and rooms; line flange classes are not on it. | yes: The plan shows where powered equipment sits and where its isolation tags are provided. | yes: The plan with its egress schedule is where exit widths against occupant load are shown. | no: Weld joints are not shown on an arrangement plan. | no: Relief devices and their set pressures are on the P&ID and the relief schedule, not on the arrangement plan. |
| **SPEC** | yes: The piping class specification tabulates flange class against design pressure. | no: A specification sheet has tables only; there is no equipment to isolate. | no: A piping or relief specification has no exits or occupant load. | yes: The piping class specification states the weld joint type and examination for each class. | yes: The relief device schedule lists each vessel with its PSV tag, set pressure and orifice. |
| **OTHER** | no: A symbols/legend or typical-detail sheet defines notation; it describes no installed item to check against this rule. | no: A symbols/legend or typical-detail sheet defines notation; it describes no installed item to check against this rule. | no: A symbols/legend or typical-detail sheet defines notation; it describes no installed item to check against this rule. | no: A symbols/legend or typical-detail sheet defines notation; it describes no installed item to check against this rule. | no: A symbols/legend or typical-detail sheet defines notation; it describes no installed item to check against this rule. |

In [2]:
for dtype, cells in lib.APPLICABILITY.items():
    print(f"{dtype}: applies = {[c for c in lib.CITATIONS if cells[c][0]]}")

ISO: applies = ['ASME B16.5 §4.2', 'ASME B31.3 §341']
PID: applies = ['ASME B16.5 §4.2', 'OSHA 1910.147', 'ASME B31.3 §322']
GA: applies = ['OSHA 1910.147', 'IBC 1005.1']
SPEC: applies = ['ASME B16.5 §4.2', 'ASME B31.3 §341', 'ASME B31.3 §322']
OTHER: applies = []


## Read the saved judge answers

The 1,200 answers of the AI run, with each drawing's type and parsed text (the text is needed to check the judge's `expected_item` against it). The timestamp is when the AI step ran.

In [3]:
raw_rows = (spark.table(RAW)
            .join(spark.table(PARSED).select("drawing_id", "parsed_markdown"), "drawing_id")
            .join(spark.table(EXTRACTED).select("drawing_id", "drawing_type"), "drawing_id")
            .collect())
judged_at = {r.drawing_id: r.judged_at for r in raw_rows}
print(f"reusing {len(raw_rows)} judge answers from {RAW}, written by the AI step at {max(judged_at.values())} UTC (job run 726850339672013); no AI call is made here")
assert len(raw_rows) == 1200 == spark.table(RAW).count(), "expected the 1,200 answers of the full AI run"

manifest = json.loads(MANIFEST.read_text())
review_dates = lib.assign_review_dates(manifest)
in_week = {d for d, v in review_dates.items() if "2026-10-31" <= v <= "2026-11-06"}
assert len(in_week) == 238
dates_df = spark.createDataFrame(list(review_dates.items()), "drawing_id string, review_date_text string").select("drawing_id", F.to_date("review_date_text").alias("review_date"))
gt_pairs = {(r.drawing_id, r.citation) for r in spark.table(GROUND_TRUTH_TABLE).collect()}
dtype_of = {r.drawing_id: r.drawing_type for r in raw_rows}
assert len(gt_pairs) == 274

reusing 1200 judge answers from mfg_mc_se_sa.bluerridge_gold_pm.compliance_judge_raw, written by the AI step at 2026-11-07 20:36:47.418214 UTC (job run 726850339672013); no AI call is made here


## Before and after the guardrails

*Before* is what the judge's answers give without the guardrails: `compliance_lib.flatten_judgement` on every answer, the function `03_compliance_check.ipynb` used (it already drops an entry whose `found_in_text` is true). *After* is `compliance_lib.judge_with_guardrails` on the same answers. Precision and recall are on `(drawing_id, citation)` against the 274 planted flaws.

In [4]:
before, after, suppressed = [], [], []
for r in raw_rows:
    before += lib.flatten_judgement(r.drawing_id, r.result, r.error)
    kept, dropped = lib.judge_with_guardrails(r.drawing_id, r.drawing_type, r.parsed_markdown, r.result, r.error)
    after += kept
    suppressed += dropped

def drawing_outcome(rows):
    by = {}
    for x in rows:
        by.setdefault(x["drawing_id"], set()).add(x["status"])
    return {d: ("FAIL" if "FAIL" in s else "REVIEW" if "REVIEW" in s else "PASS") for d, s in by.items()}

def metrics(rows):
    pairs = {(x["drawing_id"], x["citation"]) for x in rows if x["status"] == "FAIL"}
    tp = len(pairs & gt_pairs)
    out = drawing_outcome(rows)
    week = [out[d] for d in in_week]
    return {"pairs": pairs, "tp": tp, "precision": tp / max(len(pairs), 1), "recall": tp / len(gt_pairs), "outcome": out,
            "fail_rows": Counter(x["citation"] for x in rows if x["status"] == "FAIL"),
            "drawings": Counter(out.values()), "week": Counter(week)}

mb, ma = metrics(before), metrics(after)
rows = []
for label, key in (("drawings PASS", "PASS"), ("drawings FAIL", "FAIL"), ("drawings REVIEW", "REVIEW")):
    rows.append((label, mb["drawings"][key], ma["drawings"][key]))
for c in lib.CITATIONS:
    rows.append((f"FAIL rows: {c}", mb["fail_rows"][c], ma["fail_rows"][c]))
rows.append(("FAIL rows: total", sum(mb["fail_rows"].values()), sum(ma["fail_rows"].values())))
rows += [("distinct (drawing, citation) findings", len(mb["pairs"]), len(ma["pairs"])),
         ("planted flaws found (true positives)", mb["tp"], ma["tp"]),
         ("precision %", round(100 * mb["precision"], 1), round(100 * ma["precision"], 1)),
         ("recall %", round(100 * mb["recall"], 1), round(100 * ma["recall"], 1)),
         ("as-of week: reviewed", 238, 238),
         ("as-of week: passed", mb["week"]["PASS"], ma["week"]["PASS"]),
         ("as-of week: failed", mb["week"]["FAIL"], ma["week"]["FAIL"]),
         ("as-of week: needs review", mb["week"]["REVIEW"], ma["week"]["REVIEW"])]
display(spark.createDataFrame([(a, float(b), float(c)) for a, b, c in rows], "measure string, raw_judge double, guardrailed double"))
print(f"precision {mb['precision']:.1%} -> {ma['precision']:.1%}, recall {mb['recall']:.1%} -> {ma['recall']:.1%}; "
      f"as-of week failed {mb['week']['FAIL']} -> {ma['week']['FAIL']} of 238 (the planted story is 24)")

measure,raw_judge,guardrailed
drawings PASS,822.0,911.0
drawings FAIL,377.0,288.0
drawings REVIEW,1.0,1.0
FAIL rows: ASME B16.5 §4.2,99.0,89.0
FAIL rows: OSHA 1910.147,323.0,130.0
FAIL rows: IBC 1005.1,45.0,45.0
FAIL rows: ASME B31.3 §341,50.0,50.0
FAIL rows: ASME B31.3 §322,110.0,98.0
FAIL rows: total,627.0,412.0
"distinct (drawing, citation) findings",450.0,327.0


precision 54.9% -> 75.5%, recall 90.1% -> 90.1%; as-of week failed 52 -> 33 of 238 (the planted story is 24)


In [5]:
# What each guardrail suppressed, and what it cost in recall.
def reason_kind(entry):
    d = entry["suppression_detail"]
    if entry["suppressed_by"] == "rule_applicability":
        return "citation does not apply to the drawing type"
    if d.startswith("found_in_text"):
        return "judge's found_in_text is true"
    if d.startswith("expected_item"):
        return "expected_item is in the drawing text"
    if d.startswith("the lockout"):
        return "LOTO rule applied to a vessel"
    return "reason admits compliance"

by_kind = Counter((s["suppressed_by"], reason_kind(s)) for s in suppressed)
planted_entries = Counter((s["suppressed_by"], reason_kind(s)) for s in suppressed if (s["drawing_id"], s["citation"]) in gt_pairs)
display(spark.createDataFrame(
    [(by, kind, n, planted_entries[(by, kind)]) for (by, kind), n in sorted(by_kind.items())] + [("total", "", len(suppressed), sum(planted_entries.values()))],
    "suppressed_by string, kind string, suppressed_entries int, of_which_on_a_planted_flaw int"))
lost = sorted((mb["pairs"] & gt_pairs) - (ma["pairs"] & gt_pairs))
print(f"suppressed entries: {len(suppressed)} ({Counter(s['suppressed_by'] for s in suppressed)}); {sum(planted_entries.values())} of them sit on a (drawing, citation) that is a planted flaw")
print(f"recall cost: {len(lost)} planted flaws found before and not after (the other planted entries had a second, kept entry for the same drawing and citation, or were already counted)")
display(spark.createDataFrame([(d, c, dtype_of[d]) for d, c in lost], "drawing_id string, citation string, drawing_type string"))

print("false alarms (judged and not planted) by drawing type, before and after:")
fa_b = Counter(dtype_of[d] for d, c in mb["pairs"] - gt_pairs)
fa_a = Counter(dtype_of[d] for d, c in ma["pairs"] - gt_pairs)
display(spark.createDataFrame([(t, fa_b[t], fa_a[t]) for t in ("ISO", "PID", "GA", "SPEC", "OTHER")], "drawing_type string, false_alarms_before int, false_alarms_after int"))
print(f"ISO false alarms: {fa_b['ISO']} before, {fa_a['ISO']} after")
fa_after_cit = Counter(c for d, c in ma["pairs"] - gt_pairs)
print("false alarms that remain, by citation:", dict(fa_after_cit), "- mostly the lockout/tagout rule on P&IDs, where the parser misreads LOTO as ROTO-/LOT0-/LOTOTO-")
assert fa_a["ISO"] == 0 or fa_a["ISO"] < fa_b["ISO"]
assert ma["precision"] > mb["precision"] and ma["tp"] >= mb["tp"] - 10, "the guardrails should raise precision at a small recall cost"

suppressed_by,kind,suppressed_entries,of_which_on_a_planted_flaw
rule_applicability,citation does not apply to the drawing type,85,0
self_contradiction,LOTO rule applied to a vessel,20,2
self_contradiction,expected_item is in the drawing text,91,7
self_contradiction,judge's found_in_text is true,465,48
self_contradiction,reason admits compliance,19,3
total,,680,60


suppressed entries: 680 (Counter({'self_contradiction': 595, 'rule_applicability': 85})); 60 of them sit on a (drawing, citation) that is a planted flaw
recall cost: 0 planted flaws found before and not after (the other planted entries had a second, kept entry for the same drawing and citation, or were already counted)


drawing_id,citation,drawing_type


false alarms (judged and not planted) by drawing type, before and after:


drawing_type,false_alarms_before,false_alarms_after
ISO,45,0
PID,121,48
GA,12,9
SPEC,25,23
OTHER,0,0


ISO false alarms: 45 before, 0 after
false alarms that remain, by citation: {'OSHA 1910.147': 30, 'ASME B31.3 §322': 23, 'ASME B16.5 §4.2': 24, 'IBC 1005.1': 3} - mostly the lockout/tagout rule on P&IDs, where the parser misreads LOTO as ROTO-/LOT0-/LOTOTO-


## Where the highlight box comes from (`bbox_source`)

The app draws a rectangle over the drawing for each finding, so a FAIL row needs a page and a box. There are three sources, tried in this order:

1. **`ground_truth`**: the finding matches a planted flaw on `(drawing_id, citation)`. The box and page are the generator's own, which enclose exactly the offending element (a flange label, a table row, an empty slot where a tag belongs). **The demo's tight highlights come from here.** If a drawing has two findings with the same citation, only the first takes the planted box.
2. **`parse_element`**: otherwise, the smallest element in `parse_json` on the judged page whose text contains the judge's `evidence_text`, with its pixel box divided by the page's pixel size and written as `[x, y, w, h]` from the top-left. The sheets are 1224 x 792 pt (17 x 11 in landscape) and `ai_parse_document` renders them at 2.75 px per point, 3366 x 2178 px (checked below against the largest parser box on every drawing).
3. **`page`**: failing that, the whole page, `[0, 0, 1, 1]`.

**What this means in production.** A real drawing has no ground truth, so every highlight would come from source 2 or 3. Those boxes are coarser: a schematic parses as one large `figure` element, holding all its labels as one block, so the best a parse element can do for a flaw inside a P&ID is the figure's box. Table rows (specification sheets) and notes parse as smaller elements and are located much better. The comparison further down measures exactly this on the findings that matched the ground truth.

In [6]:
# Is the page really 3366 x 2178 px? The largest right and bottom edges among all parser boxes must not exceed it, and should come near it
# (the title block sits at the bottom right).
extent = spark.sql(f"""
    SELECT MAX(CAST(el.bbox[0].coord[2] AS DOUBLE)) AS max_x2, MAX(CAST(el.bbox[0].coord[3] AS DOUBLE)) AS max_y2, COUNT(*) AS elements
    FROM {PARSED}
    LATERAL VIEW explode(from_json(get_json_object(parse_json, '$.document.elements'),
        'array<struct<bbox:array<struct<coord:array<double>,page_id:int>>>>')) t AS el
""").first()
W, H = lib.SHEET_PX
print(f"{extent.elements} parser elements; largest x2 = {extent.max_x2:.0f} px (page width {W:.0f}), largest y2 = {extent.max_y2:.0f} px (page height {H:.0f})")
assert extent.max_x2 <= W and extent.max_y2 <= H and extent.max_x2 > 0.9 * W and extent.max_y2 > 0.9 * H

24924 parser elements; largest x2 = 3342 px (page width 3366), largest y2 = 2141 px (page height 2178)


## Findings with their boxes, rebuilt from the guardrailed rows

The same build as in `03_compliance_check.ipynb`: join the rows to the drawing metadata, the ground truth (left join on `(drawing_id, citation)`), the review dates and the parsed JSON, and give each FAIL row a box (`ground_truth`, else `parse_element`, else `page`; see that notebook for the rules and their limits). Only the rows come from `judge_with_guardrails` instead of `flatten_judgement`.

In [7]:
FLAT_SCHEMA = ("drawing_id string, finding_id string, status string, reason string, citation string, severity string, "
               "page_num int, evidence_text string, judged_at timestamp")
findings = spark.createDataFrame(
    [(*(r[k] for k in ("drawing_id", "finding_id", "status", "reason", "citation", "severity", "page_num", "evidence_text")), judged_at[r["drawing_id"]]) for r in after],
    FLAT_SCHEMA)
docs_all = docs = spark.table(EXTRACTED).select("drawing_id")  # the verification below counts these
n_docs = docs.count()
print(f"{findings.count()} rows for {n_docs} drawings")

1324 rows for 1200 drawings


In [8]:
gt = (spark.table(GROUND_TRUTH_TABLE).select("drawing_id", "citation", F.col("page_num").alias("gt_page"), F.col("bbox").alias("gt_bbox")))
meta = (spark.table(PARSED).select("drawing_id", "volume_path", "parse_json")
        .join(spark.table(EXTRACTED).select("drawing_id", "discipline", "revision", "drawing_type"), "drawing_id"))

# Findings of one drawing with the same citation: only the first one takes the planted box.
first_of_citation = F.row_number().over(Window.partitionBy("drawing_id", "citation").orderBy("finding_id")) == 1
joined = (findings.join(meta, "drawing_id").join(gt, ["drawing_id", "citation"], "left")
          .withColumn("is_fail", F.col("status") == "FAIL")
          .withColumn("matched", F.when(F.col("is_fail"), F.col("gt_bbox").isNotNull()))
          .withColumn("use_gt", F.col("is_fail") & F.col("gt_bbox").isNotNull() & F.when(F.col("is_fail"), first_of_citation).otherwise(False))
          )

# The parse-element fallback runs on the driver for the FAIL rows that took no planted box (a few hundred small lookups).
need_box = joined.where("is_fail AND NOT use_gt").select("drawing_id", "finding_id", "parse_json", "page_num", "evidence_text").collect()
element_boxes = spark.createDataFrame(
    [(n.drawing_id, n.finding_id, lib.resolve_parse_bbox(n.parse_json, n.page_num, n.evidence_text)) for n in need_box],
    "drawing_id string, finding_id string, element_box array<double>")
joined = joined.join(element_boxes, ["drawing_id", "finding_id"], "left")
print(f"{len(need_box)} FAIL rows without a planted box; a parse element was found for {element_boxes.where('element_box IS NOT NULL').count()}")

final = (joined
    .withColumn("bbox_source", F.when(F.col("use_gt"), "ground_truth").when(F.col("element_box").isNotNull(), "parse_element").when(F.col("is_fail"), "page"))
    .withColumn("bbox_out", F.when(F.col("use_gt"), F.col("gt_bbox")).when(F.col("element_box").isNotNull(), F.col("element_box"))
                .when(F.col("is_fail"), F.array(F.lit(0.0), F.lit(0.0), F.lit(1.0), F.lit(1.0))))
    .withColumn("page_out", F.when(F.col("use_gt"), F.col("gt_page")).when(F.col("is_fail"), F.col("page_num")))
    .join(dates_df, "drawing_id")
    .select(
        "drawing_id", "finding_id", "volume_path",
        F.col("page_out").cast("int").alias("page_num"),
        F.col("bbox_out").cast("array<double>").alias("bbox"),
        "bbox_source", "discipline", "revision", "drawing_type", "status", "severity", "reason", "citation", "evidence_text",
        F.col("matched").alias("matched_ground_truth"),
        "review_date",
        F.col("judged_at").alias("generated_at"),
    ))
print("columns:", final.columns)

165 FAIL rows without a planted box; a parse element was found for 126
columns: ['drawing_id', 'finding_id', 'volume_path', 'page_num', 'bbox', 'bbox_source', 'discipline', 'revision', 'drawing_type', 'status', 'severity', 'reason', 'citation', 'evidence_text', 'matched_ground_truth', 'review_date', 'generated_at']


In [9]:
t0 = time.time()
final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
print(f"wrote {TARGET} in {time.time() - t0:.0f} s")

spark.sql(
    f"COMMENT ON TABLE {TARGET} IS 'One row per compliance finding from the ai_query judge (databricks-claude-sonnet-4-5) over the 1,200 drawings. Every drawing has at least one row: "
    "N FAIL rows, or one PASS row (also when every entry was suppressed by a guardrail), or one REVIEW row when the judge answer could not be used. Built from compliance_judge_raw by 03b_compliance_guardrails: rule-applicability and self-consistency guardrails over the judge output, with the suppressed entries in compliance_suppressed.'"
)
COLUMN_COMMENTS = {
    "drawing_id": "Drawing id; key into drawings_extracted. With finding_id the primary key",
    "finding_id": "<drawing_id>-C01, -C02, ... for FAIL rows; <drawing_id>-PASS or <drawing_id>-REVIEW for the one-row cases",
    "volume_path": "Unity Catalog Volume path of the PDF",
    "page_num": "0-based page of the violation; NULL on PASS and REVIEW rows",
    "bbox": "[x, y, w, h] as fractions (0-1) of the page, origin top-left; NULL on PASS and REVIEW rows",
    "bbox_source": "ground_truth (the generator's planted box), parse_element (smallest parsed element holding evidence_text) or page ([0,0,1,1]); NULL on PASS and REVIEW rows",
    "discipline": "From drawings_extracted", "revision": "From drawings_extracted", "drawing_type": "From drawings_extracted",
    "status": "PASS, FAIL or REVIEW (REVIEW: the judge answer was an error or could not be parsed)",
    "severity": "critical, high, medium or low (anything else the judge said is medium); NULL unless FAIL",
    "reason": "Short explanation a project manager can read",
    "citation": "The rule violated: ASME B16.5 §4.2, OSHA 1910.147, IBC 1005.1, ASME B31.3 §341 or ASME B31.3 §322; NULL unless FAIL",
    "evidence_text": "The judge's short quote from the drawing text that shows the violation; NULL unless FAIL",
    "matched_ground_truth": "TRUE when the synthetic generator planted a flaw with this citation on this drawing; NULL unless FAIL",
    "review_date": "Simulated review date; the as-of week 2026-10-31..2026-11-06 holds 238 drawings",
    "generated_at": "When the judge step ran",
}
for column, comment in COLUMN_COMMENTS.items():
    spark.sql(f"ALTER TABLE {TARGET} ALTER COLUMN {column} COMMENT {sql_literal(comment)}")
spark.sql(f"ALTER TABLE {TARGET} SET TBLPROPERTIES ('quality' = 'gold')")
display(spark.sql(f"""
    SELECT ordinal_position AS position, column_name, full_data_type AS data_type
    FROM {catalog}.information_schema.columns
    WHERE table_schema = '{prefix}_gold_pm' AND table_name = '{TARGET.split('.')[-1]}'
    ORDER BY ordinal_position
"""))

wrote mfg_mc_se_sa.bluerridge_gold_pm.compliance_findings in 7 s


position,column_name,data_type
0,drawing_id,string
1,finding_id,string
2,volume_path,string
3,page_num,int
4,bbox,array<double>
5,bbox_source,string
6,discipline,string
7,revision,string
8,drawing_type,string
9,status,string


## Write `compliance_suppressed`

Every entry a guardrail removed, with the finding's columns plus `suppressed_by` and `suppression_detail` (why). `matched_ground_truth` says whether the entry's `(drawing_id, citation)` is a planted flaw, so the recall cost of each suppression can be read off. Boxes are NULL: a suppressed entry is not drawn.

In [10]:
meta_s = (spark.table(PARSED).select("drawing_id", "volume_path")
          .join(spark.table(EXTRACTED).select("drawing_id", "discipline", "revision", "drawing_type"), "drawing_id"))
gt_keys = spark.createDataFrame(sorted(gt_pairs), "drawing_id string, citation string").withColumn("planted", F.lit(True))
sup_df = (spark.createDataFrame(
              [(s["drawing_id"], s["finding_id"], s["page_num"], s["status"], s["severity"], s["reason"], s["citation"], s["evidence_text"],
                s["suppressed_by"], s["suppression_detail"], judged_at[s["drawing_id"]]) for s in suppressed],
              "drawing_id string, finding_id string, page_num int, status string, severity string, reason string, citation string, evidence_text string, "
              "suppressed_by string, suppression_detail string, generated_at timestamp")
          .join(meta_s, "drawing_id").join(gt_keys, ["drawing_id", "citation"], "left").join(dates_df, "drawing_id")
          .select("drawing_id", "finding_id", "volume_path", "page_num", F.lit(None).cast("array<double>").alias("bbox"), F.lit(None).cast("string").alias("bbox_source"),
                  "discipline", "revision", "drawing_type", "status", "severity", "reason", "citation", "evidence_text",
                  F.coalesce("planted", F.lit(False)).alias("matched_ground_truth"), "review_date", "generated_at", "suppressed_by", "suppression_detail"))
sup_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SUPPRESSED)
_ = spark.sql(f"COMMENT ON TABLE {SUPPRESSED} IS 'FAIL entries of the compliance judge that a deterministic guardrail removed from compliance_findings (rule_applicability: the citation does not apply to the drawing type; self_contradiction: the judge output admits compliance), with the reason. Same columns as compliance_findings plus suppressed_by and suppression_detail. Nothing is dropped silently.'")
_ = spark.sql(f"ALTER TABLE {SUPPRESSED} SET TBLPROPERTIES ('quality' = 'gold')")
_ = spark.sql(f"ALTER TABLE {SUPPRESSED} ALTER COLUMN suppressed_by COMMENT 'rule_applicability or self_contradiction'")
_ = spark.sql(f"ALTER TABLE {SUPPRESSED} ALTER COLUMN suppression_detail COMMENT 'Why the entry was suppressed'")
_ = spark.sql(f"ALTER TABLE {SUPPRESSED} ALTER COLUMN finding_id COMMENT '<drawing_id>-S01, -S02, ... per drawing'")
s_chk = spark.sql(f"SELECT COUNT(*) AS rows, COUNT(DISTINCT drawing_id, finding_id) AS distinct_keys, COUNT_IF(suppressed_by NOT IN ('rule_applicability', 'self_contradiction')) AS bad_by, "
                  f"COUNT_IF(suppression_detail IS NULL) AS no_detail, COUNT_IF(matched_ground_truth) AS on_planted FROM {SUPPRESSED}").first()
print(f"{SUPPRESSED}: {s_chk.rows} rows, {s_chk.distinct_keys} distinct keys, {s_chk.on_planted} on a planted flaw")
assert s_chk.rows == len(suppressed) == s_chk.distinct_keys and s_chk.bad_by == 0 and s_chk.no_detail == 0
display(spark.sql(f"SELECT suppressed_by, citation, drawing_type, COUNT(*) AS entries FROM {SUPPRESSED} GROUP BY ALL ORDER BY suppressed_by, entries DESC LIMIT 20"))
# Every entry the guardrails took out is accounted for: kept FAIL entries + suppressed entries = what the judge reported (found_in_text-true entries included).
assert not (set(map(tuple, sup_df.select("drawing_id", "finding_id").collect())) & set(map(tuple, spark.table(TARGET).select("drawing_id", "finding_id").collect())))

mfg_mc_se_sa.bluerridge_gold_pm.compliance_suppressed: 680 rows, 680 distinct keys, 60 on a planted flaw


suppressed_by,citation,drawing_type,entries
rule_applicability,OSHA 1910.147,ISO,82
rule_applicability,ASME B31.3 §322,ISO,3
self_contradiction,OSHA 1910.147,PID,424
self_contradiction,ASME B31.3 §322,PID,138
self_contradiction,OSHA 1910.147,GA,17
self_contradiction,ASME B16.5 §4.2,PID,7
self_contradiction,ASME B16.5 §4.2,SPEC,5
self_contradiction,IBC 1005.1,GA,3
self_contradiction,ASME B16.5 §4.2,ISO,1


## Verify

The table's shape and the row rules as assertions: the ruled schema, a primary key of `(drawing_id, finding_id)`, every drawing sent has a row and none is an orphan against `drawings_extracted`, a drawing is exactly one of {FAIL rows only, one PASS row, one REVIEW row}, FAIL rows have a box inside 0-1, one of the five citations and one of the four severities, PASS and REVIEW rows have NULL in the FAIL-only columns, every REVIEW row has a reason, and (full run) exactly 238 drawings in the as-of week.

In [11]:
EXPECTED_SCHEMA = [
    ("drawing_id", "string"), ("finding_id", "string"), ("volume_path", "string"), ("page_num", "int"), ("bbox", "array<double>"),
    ("bbox_source", "string"), ("discipline", "string"), ("revision", "string"), ("drawing_type", "string"), ("status", "string"),
    ("severity", "string"), ("reason", "string"), ("citation", "string"), ("evidence_text", "string"), ("matched_ground_truth", "boolean"),
    ("review_date", "date"), ("generated_at", "timestamp"),
]
assert spark.table(TARGET).dtypes == EXPECTED_SCHEMA, spark.table(TARGET).dtypes

cites_sql = ", ".join(sql_literal(c) for c in lib.CITATIONS)
checks = spark.sql(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT drawing_id, finding_id) AS distinct_keys,
           COUNT(DISTINCT drawing_id) AS drawings,
           COUNT_IF(status NOT IN ('PASS', 'FAIL', 'REVIEW')) AS bad_status,
           COUNT_IF(status = 'FAIL' AND (bbox IS NULL OR size(bbox) <> 4 OR NOT forall(bbox, v -> v BETWEEN 0 AND 1) OR page_num IS NULL OR bbox_source IS NULL)) AS fail_bad_box,
           COUNT_IF(status = 'FAIL' AND bbox_source NOT IN ('ground_truth', 'parse_element', 'page')) AS bad_source,
           COUNT_IF(status = 'FAIL' AND (citation IS NULL OR citation NOT IN ({cites_sql}))) AS fail_bad_citation,
           COUNT_IF(status = 'FAIL' AND (severity IS NULL OR severity NOT IN ('critical', 'high', 'medium', 'low'))) AS fail_bad_severity,
           COUNT_IF(status = 'FAIL' AND (matched_ground_truth IS NULL OR reason IS NULL OR evidence_text IS NULL)) AS fail_missing_text,
           COUNT_IF(status <> 'FAIL' AND (citation IS NOT NULL OR page_num IS NOT NULL OR bbox IS NOT NULL OR bbox_source IS NOT NULL
                                           OR severity IS NOT NULL OR matched_ground_truth IS NOT NULL)) AS sentinel_not_null,
           COUNT_IF(status = 'REVIEW' AND reason IS NULL) AS review_no_reason,
           COUNT_IF(review_date IS NULL OR generated_at IS NULL OR volume_path IS NULL) AS missing_context
    FROM {TARGET}
""")
display(checks)
c = checks.first()
shape = spark.sql(f"""
    SELECT drawing_id, COUNT(*) AS n, COUNT_IF(status = 'FAIL') AS fails, COUNT_IF(status = 'PASS') AS passes, COUNT_IF(status = 'REVIEW') AS reviews
    FROM {TARGET} GROUP BY drawing_id
""")
bad_shape = shape.where("NOT ((fails >= 1 AND passes = 0 AND reviews = 0) OR (n = 1 AND passes = 1) OR (n = 1 AND reviews = 1))").count()
orphans = spark.sql(f"SELECT COUNT(*) AS n FROM {TARGET} t WHERE NOT EXISTS (SELECT 1 FROM {EXTRACTED} e WHERE e.drawing_id = t.drawing_id)").first().n
unreviewed = docs_all.join(spark.table(TARGET).select("drawing_id").distinct(), "drawing_id", "left_anti").join(docs.select("drawing_id"), "drawing_id").count()
print(f"{c.rows} rows, {c.distinct_keys} distinct (drawing_id, finding_id), {c.drawings} drawings; {bad_shape} drawings with a wrong mix of statuses; "
      f"{orphans} rows without a drawings_extracted row; {unreviewed} drawings sent but without a row")

assert c.rows == c.distinct_keys, "(drawing_id, finding_id) is not unique"
assert c.drawings == n_docs and unreviewed == 0, "every drawing sent must have at least one row"
assert orphans == 0 and bad_shape == 0 and c.bad_status == 0
assert c.fail_bad_box == 0 and c.bad_source == 0 and c.fail_bad_citation == 0 and c.fail_bad_severity == 0 and c.fail_missing_text == 0
assert c.sentinel_not_null == 0 and c.review_no_reason == 0 and c.missing_context == 0
if not MAX_ROWS:
    assert c.drawings == 1200, "the corpus has 1,200 drawings"
    week = spark.sql(f"SELECT COUNT(DISTINCT drawing_id) AS n FROM {TARGET} WHERE review_date BETWEEN DATE'2026-10-31' AND DATE'2026-11-06'").first().n
    assert week == 238, week
    review_share = spark.sql(f"SELECT COUNT(DISTINCT drawing_id) / {n_docs} AS s FROM {TARGET} WHERE status = 'REVIEW'").first().s
    assert review_share <= 0.05, f"more than 5% of the drawings ended in REVIEW ({review_share:.1%}); read compliance_judge_raw.error"
print("all checks passed")

rows,distinct_keys,drawings,bad_status,fail_bad_box,bad_source,fail_bad_citation,fail_bad_severity,fail_missing_text,sentinel_not_null,review_no_reason,missing_context
1324,1324,1200,0,0,0,0,0,0,0,0,0


1324 rows, 1324 distinct (drawing_id, finding_id), 1200 drawings; 0 drawings with a wrong mix of statuses; 0 rows without a drawings_extracted row; 0 drawings sent but without a row
all checks passed


## What a project manager sees

The headline numbers, computed from `compliance_findings` alone: how many drawings were reviewed and how they came out, which rules were violated and how often, where the boxes came from, and the as-of week.

In [12]:
# One status per drawing: FAIL if it has any FAIL row, else REVIEW, else PASS.
drawing_status = spark.sql(f"""
    SELECT drawing_id, MIN(review_date) AS review_date,
           CASE WHEN COUNT_IF(status = 'FAIL') > 0 THEN 'FAIL' WHEN COUNT_IF(status = 'REVIEW') > 0 THEN 'REVIEW' ELSE 'PASS' END AS drawing_status,
           COUNT_IF(status = 'FAIL') AS findings
    FROM {TARGET} GROUP BY drawing_id
""")
drawing_status.createOrReplaceTempView("drawing_status")
print("drawings reviewed, and how many came out PASS / FAIL / REVIEW:")
display(spark.sql("""
    SELECT COUNT(*) AS drawings_reviewed, COUNT_IF(drawing_status = 'PASS') AS passed, COUNT_IF(drawing_status = 'FAIL') AS failed,
           COUNT_IF(drawing_status = 'REVIEW') AS needs_review, SUM(findings) AS fail_rows
    FROM drawing_status"""))
print("FAIL rows by citation (and how many matched a planted flaw):")
display(spark.sql(f"""
    SELECT citation, COUNT(*) AS fail_rows, COUNT(DISTINCT drawing_id) AS drawings, COUNT_IF(matched_ground_truth) AS matched_ground_truth,
           COUNT_IF(severity = 'critical') AS critical, COUNT_IF(severity = 'high') AS high, COUNT_IF(severity = 'medium') AS medium, COUNT_IF(severity = 'low') AS low
    FROM {TARGET} WHERE status = 'FAIL' GROUP BY citation ORDER BY fail_rows DESC"""))
print("bbox_source of the FAIL rows:")
display(spark.sql(f"""
    SELECT bbox_source, COUNT(*) AS fail_rows, ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct FROM {TARGET} WHERE status = 'FAIL' GROUP BY bbox_source ORDER BY fail_rows DESC"""))
m = spark.sql(f"SELECT COUNT(*) AS fail_rows, COUNT_IF(matched_ground_truth) AS matched FROM {TARGET} WHERE status = 'FAIL'").first()
print(f"matched_ground_truth on FAIL rows: {m.matched} of {m.fail_rows} ({m.matched / max(m.fail_rows, 1):.1%})")
print("as-of week 2026-10-31..2026-11-06 (review_date), the demo's \"reviewed / passed / failed this week\":")
display(spark.sql("""
    SELECT COUNT(*) AS reviewed, COUNT_IF(drawing_status = 'PASS') AS passed, COUNT_IF(drawing_status = 'FAIL') AS failed, COUNT_IF(drawing_status = 'REVIEW') AS needs_review
    FROM drawing_status WHERE review_date BETWEEN DATE'2026-10-31' AND DATE'2026-11-06'"""))
print("status of the drawings by drawing_type:")
display(spark.sql("SELECT e.drawing_type, COUNT(*) AS drawings, COUNT_IF(d.drawing_status = 'FAIL') AS failed FROM drawing_status d JOIN " + EXTRACTED + " e USING (drawing_id) GROUP BY e.drawing_type ORDER BY drawings DESC"))

drawings reviewed, and how many came out PASS / FAIL / REVIEW:


drawings_reviewed,passed,failed,needs_review,fail_rows
1200,911,288,1,412


FAIL rows by citation (and how many matched a planted flaw):


citation,fail_rows,drawings,matched_ground_truth,critical,high,medium,low
OSHA 1910.147,130,81,60,0,130,0,0
ASME B31.3 §322,98,76,69,98,0,0,0
ASME B16.5 §4.2,89,76,52,89,0,0,0
ASME B31.3 §341,50,49,50,0,0,50,0
IBC 1005.1,45,45,42,0,45,0,0


bbox_source of the FAIL rows:


bbox_source,fail_rows,pct
ground_truth,247,60.0
parse_element,126,30.6
page,39,9.5


matched_ground_truth on FAIL rows: 273 of 412 (66.3%)
as-of week 2026-10-31..2026-11-06 (review_date), the demo's "reviewed / passed / failed this week":


reviewed,passed,failed,needs_review
238,205,33,0


status of the drawings by drawing_type:


drawing_type,drawings,failed
ISO,397,55
PID,373,119
GA,269,62
SPEC,135,52
OTHER,26,0


## Against the planted flaws (approximate)

The formal scoring is the next task; this is a first look. A finding counts as correct when the drawing has a planted flaw with the same citation (`matched_ground_truth`). **Recall** is the share of the 274 planted flaws that the judge found (on `(drawing_id, citation)`), **precision** the share of its distinct `(drawing_id, citation)` findings that are planted flaws. On a trial run only the drawings sent are counted.

In [13]:
scored = spark.sql(f"""
    WITH judged AS (SELECT DISTINCT drawing_id, citation FROM {TARGET} WHERE status = 'FAIL'),
         sent AS (SELECT DISTINCT drawing_id FROM {TARGET}),
         planted AS (SELECT g.drawing_id, g.citation FROM {GROUND_TRUTH_TABLE} g JOIN sent USING (drawing_id))
    SELECT (SELECT COUNT(*) FROM planted) AS planted_flaws,
           (SELECT COUNT(*) FROM judged) AS judged_findings,
           (SELECT COUNT(*) FROM judged j JOIN planted p USING (drawing_id, citation)) AS true_positives
""").first()
print(f"planted flaws on the drawings sent: {scored.planted_flaws}; distinct judged findings: {scored.judged_findings}; found correctly: {scored.true_positives}")
print(f"recall {scored.true_positives / max(scored.planted_flaws, 1):.1%}, precision {scored.true_positives / max(scored.judged_findings, 1):.1%}")
display(spark.sql(f"""
    WITH planted AS (SELECT g.drawing_id, g.citation FROM {GROUND_TRUTH_TABLE} g WHERE g.drawing_id IN (SELECT drawing_id FROM {TARGET})),
         judged AS (SELECT DISTINCT drawing_id, citation FROM {TARGET} WHERE status = 'FAIL')
    SELECT COALESCE(p.citation, j.citation) AS citation, COUNT(p.citation) AS planted, COUNT(j.citation) AS judged,
           COUNT_IF(p.citation IS NOT NULL AND j.citation IS NOT NULL) AS found, COUNT_IF(p.citation IS NULL) AS false_alarms, COUNT_IF(j.citation IS NULL) AS missed
    FROM planted p FULL OUTER JOIN judged j ON p.drawing_id = j.drawing_id AND p.citation = j.citation
    GROUP BY COALESCE(p.citation, j.citation) ORDER BY citation"""))
print("false alarms and misses by drawing type:")
display(spark.sql(f"""
    WITH planted AS (SELECT g.drawing_id, g.citation FROM {GROUND_TRUTH_TABLE} g WHERE g.drawing_id IN (SELECT drawing_id FROM {TARGET})),
         judged AS (SELECT DISTINCT drawing_id, citation FROM {TARGET} WHERE status = 'FAIL')
    SELECT e.drawing_type, COUNT_IF(p.citation IS NOT NULL AND j.citation IS NOT NULL) AS found, COUNT_IF(p.citation IS NULL) AS false_alarms, COUNT_IF(j.citation IS NULL) AS missed
    FROM planted p FULL OUTER JOIN judged j ON p.drawing_id = j.drawing_id AND p.citation = j.citation
    JOIN {EXTRACTED} e ON e.drawing_id = COALESCE(p.drawing_id, j.drawing_id)
    GROUP BY e.drawing_type ORDER BY e.drawing_type"""))

planted flaws on the drawings sent: 274; distinct judged findings: 327; found correctly: 247
recall 90.1%, precision 75.5%


citation,planted,judged,found,false_alarms,missed
ASME B16.5 §4.2,63,76,52,24,11
ASME B31.3 §322,58,76,53,23,5
ASME B31.3 §341,49,49,49,0,0
IBC 1005.1,45,45,42,3,3
OSHA 1910.147,59,81,51,30,8


false alarms and misses by drawing type:


drawing_type,found,false_alarms,missed
GA,59,9,8
ISO,59,0,1
PID,92,48,18
SPEC,37,23,0


## How good would the production highlight be?

For the findings that matched a planted flaw, the table holds the planted box. What the `parse_element` rule would have produced for the same finding is computed here, to show how much coarser it is: the share of those findings for which it finds an element at all, the share whose box contains the planted one, and the area of that box compared with the planted box (1.0 = as tight; a `figure` holding a whole schematic is 100 times larger or more).

In [14]:
matched_rows = (spark.table(TARGET).where("status = 'FAIL' AND bbox_source = 'ground_truth'")
                .join(spark.table(PARSED).select("drawing_id", "parse_json"), "drawing_id")
                .select("drawing_id", "drawing_type", "citation", "page_num", "bbox", "evidence_text", "parse_json").collect())
comparison = {}
for r in matched_rows:
    box = lib.resolve_parse_bbox(r.parse_json, r.page_num, r.evidence_text)
    gx, gy, gw, gh = r.bbox
    entry = comparison.setdefault(r.drawing_type, {"findings": 0, "located": 0, "contains": 0, "ratios": []})
    entry["findings"] += 1
    if box:
        entry["located"] += 1
        x, y, w, h = box
        if x - 0.002 <= gx and y - 0.002 <= gy and x + w + 0.002 >= gx + gw and y + h + 0.002 >= gy + gh:
            entry["contains"] += 1
        entry["ratios"].append((w * h) / max(gw * gh, 1e-9))
rows = []
for dtype, e in sorted(comparison.items()):
    ratios = sorted(e["ratios"])
    rows.append((dtype, e["findings"], e["located"], e["contains"], round(ratios[len(ratios) // 2], 1) if ratios else None))
display(spark.createDataFrame(rows, "drawing_type string, matched_findings int, element_found int, element_contains_planted_box int, median_area_vs_planted double"))
tot = [sum(x[i] for x in rows) for i in (1, 2, 3)]
print(f"{tot[0]} matched findings: a parse element holds the evidence text for {tot[1]}, and its box contains the planted box for {tot[2]}")

drawing_type,matched_findings,element_found,element_contains_planted_box,median_area_vs_planted
GA,59,33,33,93.0
ISO,59,59,59,140.6
PID,92,92,44,6.5
SPEC,37,11,5,6.3


247 matched findings: a parse element holds the evidence text for 195, and its box contains the planted box for 141


## A sample for the repository

Twenty rows written to `outputs/sample_findings.jsonl`, one JSON object per line: FAIL rows first (two per citation where the judge found that many, so all five rules appear), then PASS rows, and every REVIEW row there is (up to three), filled up with more FAIL rows.

In [15]:
import datetime
final_rows = spark.table(TARGET).orderBy("drawing_id", "finding_id").collect()
fails = [r for r in final_rows if r.status == "FAIL"]
passes = [r for r in final_rows if r.status == "PASS"]
reviews = [r for r in final_rows if r.status == "REVIEW"][:3]

picked = []
for citation in lib.CITATIONS:  # up to two FAIL rows per rule, one of them matched to the ground truth when possible
    of_rule = [r for r in fails if r.citation == citation]
    first = next((r for r in of_rule if r.matched_ground_truth and r.bbox_source == "ground_truth"), None)
    chosen = ([first] if first else []) + [r for r in of_rule if r is not first][:2 - bool(first)]
    picked += chosen
picked += reviews
picked += passes[:8]
picked_keys = {(r.drawing_id, r.finding_id) for r in picked}
picked += [r for r in fails if (r.drawing_id, r.finding_id) not in picked_keys][: max(20 - len(picked), 0)]
picked = picked[:20]
if not MAX_ROWS:
    assert len(picked) == 20 and {r.citation for r in picked if r.status == "FAIL"} == set(lib.CITATIONS), "expected 20 rows covering all five rules"

utc = spark.conf.get("spark.sql.session.timeZone") in ("UTC", "Etc/UTC")  # serverless sessions run in UTC
lines = []
for r in picked:
    d = r.asDict()
    d["bbox"] = list(r.bbox) if r.bbox is not None else None
    d["review_date"] = r.review_date.isoformat()
    d["generated_at"] = r.generated_at.strftime("%Y-%m-%dT%H:%M:%S") + ("Z" if utc else "")
    lines.append(json.dumps(d, ensure_ascii=False))
OUT.mkdir(exist_ok=True)
(OUT / "sample_findings.jsonl").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"wrote {OUT / 'sample_findings.jsonl'}: {len(lines)} rows ({sum(r.status == 'FAIL' for r in picked)} FAIL, {sum(r.status == 'PASS' for r in picked)} PASS, {sum(r.status == 'REVIEW' for r in picked)} REVIEW)\n")
print("\n".join(lines[:6]))

wrote outputs/sample_findings.jsonl: 20 rows (11 FAIL, 8 PASS, 1 REVIEW)

{"drawing_id": "BRL-INS-PID-0052", "finding_id": "BRL-INS-PID-0052-C01", "volume_path": "/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/BRL-INS-PID-0052.pdf", "page_num": 0, "bbox": [0.235, 0.5532, 0.0334, 0.0365], "bbox_source": "ground_truth", "discipline": "instr", "revision": "B", "drawing_type": "PID", "status": "FAIL", "severity": "critical", "reason": "Line 10-BOG-2551-B3S has design pressure 500 PSIG and uses flange class CL150, whose maximum allowable pressure is 275 PSIG, below the design pressure.", "citation": "ASME B16.5 §4.2", "evidence_text": "FLG CL150", "matched_ground_truth": true, "review_date": "2026-07-12", "generated_at": "2026-11-07T20:36:47Z"}
{"drawing_id": "BRL-INS-PID-0055", "finding_id": "BRL-INS-PID-0055-C01", "volume_path": "/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/BRL-INS-PID-0055.pdf", "page_num": 0, "bbox": [0.2299, 0.2501, 0.0334, 0.0365], "bbox_source": "ground_truth